# wifi constelation 5
most systems as standard start at 80mhz bandwidth so this will mostlikely do nothing for you in data terms 

do not use unless you apply the fixes following
https://developer.analog.com/docs/system-level/solutions/platforms/phaser/setup/pluto-setup/index.html
im pretty certain that analog garante nothing when it comes to plutos accepting these also ensure you have a rev c for these settings (probably do its ubiqutous but some have rev a and b or d which will have issues)
also ensure you have firmware 0.39

In [8]:
import io
import threading
import time
import traceback
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Headless renderer, completely thread-safe
import matplotlib.pyplot as plt
import adi
from IPython.display import display
import ipywidgets as widgets

# Units & Constants
us = 1e-6
KHz = 1e3
MHz = 1e6
GHz = URI = 1e9

# --- Channel & Bandwidth Definitions ---
# 20 MHz Channels
wifi5_channels_20 = [36, 40, 44, 48, 149, 153, 157, 161, 165]
wifi5_freqs_20 = [(ch * 5 + 5000) * 1e6 for ch in wifi5_channels_20]

# 40 MHz Bonded Channels (Center frequencies of 40 MHz Wi-Fi channels)
wifi5_channels_40 = [38, 46, 54, 62, 151, 159]
wifi5_freqs_40 = [
    (38 * 5 + 5000) * 1e6,
    (46 * 5 + 5000) * 1e6,
    (54 * 5 + 5000) * 1e6,
    (62 * 5 + 5000) * 1e6,
    (151 * 5 + 5000) * 1e6,
    (159 * 5 + 5000) * 1e6
]

# --- UI Widgets ---
slider_layout = widgets.Layout(width='600px')

# Bandwidth selector dropdown
bw_dropdown = widgets.Dropdown(
    options=[("20 MHz Bandwidth", 20), ("40 MHz Bandwidth", 40)],
    value=20,
    description="BW Mode:",
    disabled=False
)

# Initial channel options set to 20 MHz
initial_options = [(f"Channel {ch} ({freq / 1e6:.0f} MHz)", freq) for ch, freq in zip(wifi5_channels_20, wifi5_freqs_20)]
channel_dropdown = widgets.Dropdown(
    options=initial_options,
    value=wifi5_freqs_20[0],
    description="5 GHz Ch:",
    disabled=False
)

# Callback to dynamically update channel options when BW mode changes
def on_bw_change(change):
    if change['new'] == 40:
        new_options = [(f"Channel {ch} (40MHz bonded @ {freq / 1e6:.0f} MHz)", freq) for ch, freq in zip(wifi5_channels_40, wifi5_freqs_40)]
        channel_dropdown.options = new_options
        channel_dropdown.value = wifi5_freqs_40[0]
    else:
        new_options = [(f"Channel {ch} ({freq / 1e6:.0f} MHz)", freq) for ch, freq in zip(wifi5_channels_20, wifi5_freqs_20)]
        channel_dropdown.options = new_options
        channel_dropdown.value = wifi5_freqs_20[0]

bw_dropdown.observe(on_bw_change, names='value')

gain_slider = widgets.IntSlider(value=40, min=0, max=73, step=1, description="RX Gain (dB):")
gi_dropdown = widgets.Dropdown(options=[("0.8 µs", 0), ("1.6 µs", 1), ("3.2 µs", 2)], value=0, description="Guard Int:")
start_button = widgets.ToggleButton(value=False, description="Start Stream", button_style="success", icon="play")
update_period = widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.1, description='Period (s):', style={'description_width': 'initial'}, layout=slider_layout)
status_label = widgets.Label(value="Status: Idle")
output_image = widgets.Image(format="png", width=800, height=400)
constellation_image = widgets.Image(format="png", width=400, height=400)

dashboard_layout = widgets.VBox([
    widgets.HBox([bw_dropdown, channel_dropdown, gain_slider]),
    widgets.HBox([gi_dropdown, start_button, status_label]),
    update_period,
    widgets.HBox([output_image, constellation_image])
])

# --- Helper Functions ---
def render_plot_bytes(x_data, psd, center_freq, bw_mhz):
    fig, ax = plt.subplots(figsize=(10, 4), dpi=80)
    ax.plot(x_data, psd, color='cyan', linewidth=1)
    ax.set_facecolor('#111111')
    ax.set_title(f"PlutoSDR Live PSD @ {center_freq / MHz:.1f} MHz (BW: {bw_mhz} MHz)", color='white')
    ax.set_xlabel("Frequency (MHz)", color='white')
    ax.set_ylabel("Relative Power (dB)", color='white')
    ax.tick_params(colors='white')
    ax.grid(True, which='both', linestyle='--', alpha=0.4)
    ax.set_ylim(-20, 80)
    fig.patch.set_facecolor('#1c1c1c')
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return buf.getvalue()

def render_constellation_bytes(data_iq):
    fig, ax = plt.subplots(figsize=(4, 4), dpi=80)
    if data_iq is not None and len(data_iq) > 0:
        ax.scatter(data_iq.real, data_iq.imag, color='magenta', s=5, alpha=0.6)
    ax.set_facecolor('#111111')
    ax.set_title("OFDM Data Constellation", color='white')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.tick_params(colors='white')
    ax.grid(True, linestyle='--', alpha=0.3)
    fig.patch.set_facecolor('#1c1c1c')
    plt.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    buf.seek(0)
    return buf.getvalue()

# --- Streaming Thread Loop ---
stream_thread = None
is_streaming = False

def rx_stream_worker():
    global is_streaming
    try:
        status_label.value = "Status: Connecting to Pluto..."
        
        sdr = adi.Pluto("ip:192.168.2.1")
        
        # Set sample rate based on selected bandwidth mode (20 MHz or 40 MHz)
        current_bw_mhz = bw_dropdown.value
        samplerate = int(current_bw_mhz * MHz)
        sdr.sample_rate = samplerate
        sdr.rx_rf_bandwidth = int(samplerate)
        
        try:
            sdr.tx_destroy_buffer()
        except Exception:
            pass
            
        sdr.rx_lo = int(channel_dropdown.value)
        sdr.rx_rf_gain_control_mode = "manual"
        sdr.rx_hardwaregain_chan0 = int(gain_slider.value)
        
        sdr.tx_lo = int(channel_dropdown.value)
        sdr.tx_rf_gain_control_mode = "manual"
        sdr.tx_hardwaregain_chan0 = 0
        sdr.tx_cyclic_buffer = True

        sdr.tx(np.zeros(4096, dtype=np.complex64))
        sdr.rx_buffer_size = 1024 * 32
        
        status_label.value = f"Status: Streaming {current_bw_mhz}MHz mode on {sdr.rx_lo / 1e6} MHz..."
        
        stf_period = 16 
        
        fft_size = 256
        subcarrier_space = 78.125 * KHz
        symbol_duration = 12.8 * us
        symbol_samples = int(round(symbol_duration * samplerate))
        gi_samples = [int(round(gi * samplerate)) for gi in [0.8 * us, 1.6 * us, 3.2 * us]]

        pilot_indices = np.array([-116, -90, -48, -22, 22, 48, 90, 116])
        active_indices = np.concatenate([np.arange(-122, -1), np.arange(2, 123)])
        data_indices = np.setdiff1d(active_indices, pilot_indices)
        data_fft_bins = (data_indices + fft_size) % fft_size
        
        while is_streaming:
            # Dynamic updates from UI sliders
            target_samplerate = int(bw_dropdown.value * MHz)
            if sdr.sample_rate != target_samplerate:
                sdr.sample_rate = target_samplerate
                sdr.rx_rf_bandwidth = target_samplerate
                samplerate = target_samplerate

            sdr.rx_lo = int(channel_dropdown.value)
            sdr.rx_hardwaregain_chan0 = int(gain_slider.value)
            sdr.tx_lo = int(channel_dropdown.value)
            
            samples = sdr.rx()
            if samples is None or len(samples) == 0:
                continue
                
            if len(samples) > stf_period * 4:
                fft_vals = np.fft.fftshift(np.fft.fft(samples[:4096]))
                psd = 20 * np.log10(np.abs(fft_vals) + 1e-6)
                freqs = (np.fft.fftshift(np.fft.fftfreq(4096, 1 / samplerate)) + sdr.rx_lo) / 1e6
                output_image.value = render_plot_bytes(freqs, psd, sdr.rx_lo, bw_dropdown.value)
                
                window_size = stf_period * 4
                correlation_profile = np.convolve(np.abs(samples), np.ones(window_size), mode='valid')
                
                if len(correlation_profile) > 0:
                    sync_idx = np.argmax(correlation_profile)
                    selected_gi_len = gi_samples[gi_dropdown.value]
                    total_block_len = selected_gi_len + symbol_samples
                    
                    if sync_idx + total_block_len <= len(samples):
                        aligned_block = samples[sync_idx : sync_idx + total_block_len]
                        freq_bins = np.fft.fft(aligned_block[:fft_size], n=fft_size)
                        data_iq = freq_bins[data_fft_bins]
                        
                        if data_iq is not None and len(data_iq) > 0:
                            phase_offset = np.angle(np.mean(data_iq))
                            data_iq = data_iq * np.exp(-1j * phase_offset)
                            
                        constellation_image.value = render_constellation_bytes(data_iq)
                    
            time.sleep(update_period.value)
            
        try:
            sdr.tx_destroy_buffer()
        except Exception:
            pass
            
        status_label.value = "Status: Stopped"
    except Exception as e:
        traceback.print_exc()
        status_label.value = f"Status: Error - {str(e)}"
        start_button.value = False

def on_start_button_change(change):
    global is_streaming, stream_thread
    if change['new']:
        is_streaming = True
        status_label.value = "Status: Initializing PlutoSDR..."
        stream_thread = threading.Thread(target=rx_stream_worker, daemon=True)
        stream_thread.start()
    else:
        is_streaming = False
        status_label.value = "Status: Stopping..."

start_button.observe(on_start_button_change, names='value')

# Seed an initial blank plot
output_image.value = render_plot_bytes(np.linspace(5175, 5185, 1024), np.zeros(1024), 5180e6, 20)

# --- Display Dashboard ---
display(dashboard_layout)